# MedGuide Retrieval & Ranking Experiment

Interactive inspection of the persisted **real** BM25, Dense, and Hybrid retrieval run. This notebook does not rerun models or invent missing scores. Human labels are stored separately in `relevance_annotations.json`; `not_reviewed` is never treated as irrelevant.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display, Markdown

def find_engine_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'artifacts' / 'retrieval-experiment' / 'ranking_results.json').exists():
            return candidate
        nested = candidate / 'medguide-evidence-engine'
        if (nested / 'artifacts' / 'retrieval-experiment' / 'ranking_results.json').exists():
            return nested
    raise FileNotFoundError('Could not locate medguide-evidence-engine')

ROOT = find_engine_root()
ARTIFACTS = ROOT / 'artifacts' / 'retrieval-experiment'
rankings_payload = json.loads((ARTIFACTS / 'ranking_results.json').read_text())
rankings = rankings_payload['rankings']
metadata = json.loads((ARTIFACTS / 'model_metadata.json').read_text())
metrics = json.loads((ARTIFACTS / 'evaluation_metrics.json').read_text())
annotations_path = ARTIFACTS / 'relevance_annotations.json'
annotations = json.loads(annotations_path.read_text())
by_query = {item['query_id']: item for item in rankings}
print(f'Loaded {len(rankings)} questions from run {rankings_payload["run_id"]}')
print('Artifacts:', ARTIFACTS)

Loaded 140 questions from run 1ca01cbd85edbb91
Artifacts: /Users/elenadalbonidarocha/Documents/medguide/medguide_web/medguide-evidence-engine/artifacts/retrieval-experiment


In [2]:
overview = pd.DataFrame([{
    'questions': len(rankings),
    'corpus_documents': rankings_payload['corpus'].get('document_count'),
    'valid_chunks': rankings_payload['corpus'].get('chunk_count'),
    'embedding_model': metadata['embedding']['model_id'],
    'embedding_dimensions': metadata['embedding']['dimension'],
    'normalized': metadata['embedding']['normalized'],
    'reranker': metadata['reranker']['model_id'],
    'evaluable_questions': metrics['evaluable_question_count'],
}])
display(overview.T.rename(columns={0: 'value'}))
if metrics['evaluable_question_count'] == 0:
    display(Markdown('> Ranking metrics are **not yet available**: no question has sufficiently complete human relevance judgments.'))

,value
questions,140
corpus_documents,None
valid_chunks,352
embedding_model,intfloat/multilingual-e5-base
embedding_dimensions,768
normalized,True
reranker,sentence-transformers/paraphrase-multilingual-...
evaluable_questions,0


> Ranking metrics are **not yet available**: no question has sufficiently complete human relevance judgments.

In [3]:
latency_rows = []
for query in rankings:
    for method, run in query['methods'].items():
        latency_rows.append({'method': method, 'latency_ms': run['latency_ms']})
latencies = pd.DataFrame(latency_rows)
display(latencies.groupby('method')['latency_ms'].agg(['count', 'mean', 'median', 'min', 'max']).round(2))

,count,mean,median,min,max
method,,,,,
bm25,140,1.26,1.18,0.56,5.07
dense,140,34.74,31.95,24.31,164.49
hybrid,140,13.79,12.50,10.59,40.61


In [4]:
DISPLAY_COLUMNS = ['rank', 'pmid', 'title', 'study_type', 'section', 'bm25_score', 'vector_similarity', 'rrf_score', 'reranker_score', 'source_url', 'chunk_text']

def show_question(query_id, top_k=10):
    query = by_query[query_id]
    display(Markdown(f"## {query_id}\n**Question:** {query['original_question']}  \n**Normalized query:** {query['normalized_query']}  \n**Category / intent / language:** {query['category']} / {query['intent']} / {query['language']}"))
    for method in ['bm25', 'dense', 'hybrid']:
        run = query['methods'][method]
        display(Markdown(f"### {method.upper()} — {run['latency_ms']:.2f} ms"))
        frame = pd.DataFrame(run['results'][:top_k])
        for column in DISPLAY_COLUMNS:
            if column not in frame:
                frame[column] = pd.NA
        frame['study_type'] = frame['study_type'].apply(lambda value: ', '.join(value) if isinstance(value, list) else value)
        display(frame[DISPLAY_COLUMNS].style.format(na_rep='unavailable'))

query_ids = list(by_query)
show_question(query_ids[0])

## GQ-SUP-061-01
**Question:** Does Collagen Peptides actually improve skin elasticity?  
**Normalized query:** Does Collagen Peptides actually improve skin elasticity?  
**Category / intent / language:** Beauty / effectiveness / en

### BM25 — 2.13 ms

,rank,pmid,title,study_type,section,bm25_score,vector_similarity,rrf_score,reranker_score,source_url,chunk_text
0,1,37822045,"Low-molecular-weight collagen peptides supplement promotes a healthy skin: A randomized, double-blinded, placebo-controlled study.","Randomized Controlled Trial, Journal Article",Abstract,33.823849,0.886454,unavailable,unavailable,https://pubmed.ncbi.nlm.nih.gov/37822045/,"BACKGROUND: Oral collagen peptides supplementation was reported to improve skin integrity and counteract skin aging. AIMS: A randomized, double-blinded, placebo-controlled study was conducted to clinically evaluate the impact of low-molecular-weight collagen peptides on the human skin. PATIENTS/METHODS: Healthy adult participants (n = 100) were randomly assigned to receive a test product containing low-molecular-weight collagen peptides or a placebo. Parameters of skin wrinkles, elasticity, hydration, and whitening (melanin and erythema indexes) were measured at baseline and after 4, 8, and 12 weeks. RESULTS: Compared with the placebo group, the average skin roughness, maximum of all peak-to-valley values, maximum peak height of the wrinkle, and average maximum height of the wrinkle were significantly improved in the test group. Parameters of skin elasticity, including overall elasticity, net elasticity, and biological elasticity, were also significantly improved in the test group at Week 12 as compared with the placebo group. Moreover, skin hydration and whitening parameters changed more significantly in the test group than in the placebo group. None of the participants experienced adverse events related to the test product. CONCLUSIONS: Taken together, these findings suggest that low-molecular-weight collagen peptides supplementation can safely ehance human skin wrinkling, hydration, elasticity, and whitening properties."
1,2,40911749,"Oral Hyaluronic Acid Supplement: Efficacy in Skin Hydration, Elasticity, and Wrinkle Depth Reduction.","Journal Article, Meta-Analysis",Abstract,12.763413,0.827111,unavailable,unavailable,https://pubmed.ncbi.nlm.nih.gov/40911749/,"BACKGROUND: Hyaluronic acid (HA) is a key component naturally present in the skin and is commonly used in skincare formulations and dermal fillers. Recently, there has been growing interest in oral HA supplementation for maintaining skin health and preventing aging due to its potential systemic benefits. OBJECTIVE: This study aims to provide a comprehensive analysis of the efficacy of oral HA supplementation on various skin parameters, including hydration, elasticity, firmness, wrinkle reduction, and transepidermal water loss. METHOD: Electronic databases (PubMed and Google Scholar) were searched for relevant articles. A total of 7 randomized controlled trials (RCTs) on the use of oral HA as a supplement were identified. A meta-analysis was performed to compare the standardized mean differences between the treatment (HA) and control (placebo) groups. RESULTS: The meta-analysis revealed statistically significant improvements in skin hydration, elasticity, and wrinkle depth following oral HA supplementation. Although the effects of oral HA on skin firmness, wrinkle volume, and transepidermal water loss were not statistically significant, a general trend of improvement was observed in these parameters. CONCLUSION: The findings of this analysis align with existing literature, underscoring the potential of oral HA supplementation as a valuable component in anti-aging and skincare regimens. However, the study's limited sample size and heterogeneity among included studies call for larger, more robust trials to confirm these findings and further explore the effects of oral HA on skin health."
2,3,36546868,"Effects of 8 Weeks of Shilajit Supplementation on Serum Pro-c1α1, a Biomarker of Type 1 Collagen Synthesis: A Randomized Control Trial.","Randomized Controlled Trial, Journal Article",Abstract,8.829533,0.839673,unavailable,unavailable,https://pubmed.ncbi.nlm.nih.gov/36546868/,"Typ

### DENSE — 49.01 ms

,rank,pmid,title,study_type,section,bm25_score,vector_similarity,rrf_score,reranker_score,source_url,chunk_text
0,1,37822045,"Low-molecular-weight collagen peptides supplement promotes a healthy skin: A randomized, double-blinded, placebo-controlled study.","Randomized Controlled Trial, Journal Article",Abstract,33.823849,0.886454,unavailable,unavailable,https://pubmed.ncbi.nlm.nih.gov/37822045/,"BACKGROUND: Oral collagen peptides supplementation was reported to improve skin integrity and counteract skin aging. AIMS: A randomized, double-blinded, placebo-controlled study was conducted to clinically evaluate the impact of low-molecular-weight collagen peptides on the human skin. PATIENTS/METHODS: Healthy adult participants (n = 100) were randomly assigned to receive a test product containing low-molecular-weight collagen peptides or a placebo. Parameters of skin wrinkles, elasticity, hydration, and whitening (melanin and erythema indexes) were measured at baseline and after 4, 8, and 12 weeks. RESULTS: Compared with the placebo group, the average skin roughness, maximum of all peak-to-valley values, maximum peak height of the wrinkle, and average maximum height of the wrinkle were significantly improved in the test group. Parameters of skin elasticity, including overall elasticity, net elasticity, and biological elasticity, were also significantly improved in the test group at Week 12 as compared with the placebo group. Moreover, skin hydration and whitening parameters changed more significantly in the test group than in the placebo group. None of the participants experienced adverse events related to the test product. CONCLUSIONS: Taken together, these findings suggest that low-molecular-weight collagen peptides supplementation can safely ehance human skin wrinkling, hydration, elasticity, and whitening properties."
1,2,37551682,"Collagen peptides supplementation improves function, pain, and physical and mental outcomes in active adults.","Randomized Controlled Trial, Journal Article",Abstract,6.898567,0.849747,unavailable,unavailable,https://pubmed.ncbi.nlm.nih.gov/37551682/,"INTRODUCTION: Chronic pain affects 19% of adults in the United States, with increasing prevalence in active and aging populations. Pain can limit physical activity and activities of daily living (ADLs), resulting in declined mental and social health. Nutritional interventions for pain currently target inflammation or joint health, but few influence both. Collagen, the most abundant protein in the human body and constituent of the extra cellular matrix, is such a nutraceutical. While there have been reports of reductions in pain with short-term collagen peptide (CP) supplementation, there are no long-term studies specifically in healthy middle-aged active adults. PURPOSE: To determine the effects of daily CP consumption over 3, 6, and 9 months on survey measures of pain, function, and physical and mental health using The Knee Injury & Osteoarthritis Outcomes Score (KOOS) and Veterans Rand 12 (VR-12) in middle-aged active adults. METHODS: This study was a double-blind randomized control trial with three treatment groups (Placebo, 10 g/d CP, and 20 g/d CP). RESULTS: Improvements in ADLs (p = .031, ηp2 = .096) and pain (p = .037, ηp2 = .164) were observed with 10 g/d CP over 6 months, although pain only improved in high frequency exercisers (>180 min/week). Additionally, VR-12 mental component scores (MCS) improved with 10 g/d of CP over 3-9 months (p = .017, ηp2 = .309), while physical component scores (PCS) improved with 20 g/d of CP over 3-9 months, but only in females (p = .013, ηp2= .582). CONCLUSION: These findings suggest 10 to 20 g/d of CP supplementation over 6 to 9 months may improve ADLs, pain, MCS, and PCS in middle-aged active adults."
2,3,36546868,"Effects of 8 Weeks of Shilajit Supplementation on Serum Pro-c1α1, a Biomarker of Type 1 Collagen Synthesis: A Randomized Control Trial.","Randomized Controlled Trial, Journal Article",Abstract,8.8295

### HYBRID — 15.64 ms

,rank,pmid,title,study_type,section,bm25_score,vector_similarity,rrf_score,reranker_score,source_url,chunk_text
0,1,42696639,Effects of essential amino acid supplementation on musculotendinous recovery following eccentric plantar flexor exercise: a randomized controlled trial.,"Journal Article, Randomized Controlled Trial",Abstract,0.000000,0.824429,0.013889,0.404898,https://pubmed.ncbi.nlm.nih.gov/42696639/,"soreness at 24 hours compared with placebo. Calf muscle thickness increased following exercise in both groups; however, swelling returned to baseline by 72 hours in the EAA group but remained elevated in placebo (Condition × Time, p < 0.001). Achilles tendon thickness decreased immediately post-exercise (p = 0.005) but was not influenced by supplementation. CONCLUSIONS: EAA supplementation modestly reduced soreness and was associated with faster recovery of muscle swelling but did not influence tendon morphology within 72 hours."
1,2,37551682,"Collagen peptides supplementation improves function, pain, and physical and mental outcomes in active adults.","Randomized Controlled Trial, Journal Article",Abstract,6.898567,0.849747,0.031054,0.343365,https://pubmed.ncbi.nlm.nih.gov/37551682/,"INTRODUCTION: Chronic pain affects 19% of adults in the United States, with increasing prevalence in active and aging populations. Pain can limit physical activity and activities of daily living (ADLs), resulting in declined mental and social health. Nutritional interventions for pain currently target inflammation or joint health, but few influence both. Collagen, the most abundant protein in the human body and constituent of the extra cellular matrix, is such a nutraceutical. While there have been reports of reductions in pain with short-term collagen peptide (CP) supplementation, there are no long-term studies specifically in healthy middle-aged active adults. PURPOSE: To determine the effects of daily CP consumption over 3, 6, and 9 months on survey measures of pain, function, and physical and mental health using The Knee Injury & Osteoarthritis Outcomes Score (KOOS) and Veterans Rand 12 (VR-12) in middle-aged active adults. METHODS: This study was a double-blind randomized control trial with three treatment groups (Placebo, 10 g/d CP, and 20 g/d CP). RESULTS: Improvements in ADLs (p = .031, ηp2 = .096) and pain (p = .037, ηp2 = .164) were observed with 10 g/d CP over 6 months, although pain only improved in high frequency exercisers (>180 min/week). Additionally, VR-12 mental component scores (MCS) improved with 10 g/d of CP over 3-9 months (p = .017, ηp2 = .309), while physical component scores (PCS) improved with 20 g/d of CP over 3-9 months, but only in females (p = .013, ηp2= .582). CONCLUSION: These findings suggest 10 to 20 g/d of CP supplementation over 6 to 9 months may improve ADLs, pain, MCS, and PCS in middle-aged active adults."
2,3,37822045,"Low-molecular-weight collagen peptides supplement promotes a healthy skin: A randomized, double-blinded, placebo-controlled study.","Randomized Controlled Trial, Journal Article",Abstract,33.823849,0.886454,0.032787,0.329212,https://pubmed.ncbi.nlm.nih.gov/37822045/,"BACKGROUND: Oral collagen peptides supplementation was reported to improve skin integrity and counteract skin aging. AIMS: A randomized, double-blinded, placebo-controlled study was conducted to clinically evaluate the impact of low-molecular-weight collagen peptides on the human skin. PATIENTS/METHODS: Healthy adult participants (n = 100) were randomly assigned to receive a test product containing low-molecular-weight collagen peptides or a placebo. Parameters of skin wrinkles, elasticity, hydration, and whitening (melanin and erythema indexes) were measured at baseline and after 4, 8, and 12 weeks. RESULTS: Compared with the placebo group, the average skin roughness, maximum of all peak-to-valley values, maximum peak height of the wrinkle, and average maximum height of the wrinkle were significantly improved in the test gr

In [5]:
# Optional selector. If ipywidgets is unavailable, call show_question('QUERY_ID') manually.
try:
    import ipywidgets as widgets
    widgets.interact(lambda query_id: show_question(query_id), query_id=widgets.Dropdown(options=query_ids, description='Question'))
except ImportError:
    print('ipywidgets is unavailable. Choose an ID from query_ids and call show_question(query_id).')

interactive(children=(Dropdown(description='Question', options=('GQ-SUP-061-01', 'GQ-SUP-062-02', 'GQ-SUP-063-…

In [6]:
def top_document_ids(query, method, k=10):
    return [row['source_document_id'] for row in query['methods'][method]['results'][:k]]

overlap_rows = []
for query in rankings:
    bm25, dense, hybrid = (set(top_document_ids(query, method)) for method in ['bm25', 'dense', 'hybrid'])
    overlap_rows.append({
        'query_id': query['query_id'],
        'question': query['original_question'],
        'bm25_dense_overlap_at_10': len(bm25 & dense),
        'bm25_hybrid_overlap_at_10': len(bm25 & hybrid),
        'dense_hybrid_overlap_at_10': len(dense & hybrid),
    })
overlap = pd.DataFrame(overlap_rows)
display(overlap.describe().round(2))
display(overlap.sort_values('bm25_dense_overlap_at_10').head(20))

,bm25_dense_overlap_at_10,bm25_hybrid_overlap_at_10,dense_hybrid_overlap_at_10
count,140.00,140.00,140.00
mean,2.83,4.71,5.08
std,1.26,1.15,1.09
min,0.00,2.00,2.00
25%,2.00,4.00,4.00
50%,3.00,5.00,5.00
75%,4.00,5.00,6.00
max,6.00,7.00,8.00


,query_id,question,bm25_dense_overlap_at_10,bm25_hybrid_overlap_at_10,dense_hybrid_overlap_at_10
34,GQ-SUP-097-09,What is known about long-term safety of Fadogi...,0,6,4
62,GQ-SUP-037-11,Which formulations and routes of Postbiotics w...,0,5,4
7,GQ-SUP-068-08,"Has MSM been studied in older adults, children...",0,4,6
90,GQ-SUP-007-13,"How strong is the human evidence for Fisetin, ...",0,2,6
25,GQ-SUP-011-19,Помогает ли Lion’s Mane для цели «improve memo...,1,5,4
24,GQ-SUP-011-16,does lions mane suplement really work,1,4,4
105,GQ-SUP-048-15,"Across all recent human studies, what is consi...",1,2,6
48,GQ-SUP-057-10,Which studies found no statistically significa...,1,4,5
51,GQ-SUP-060-13,How strong is the human evidence for Electroly...,1,5,4
30,GQ-SUP-093-05,How long was Ergothioneine used in human studi...,1,3,7


In [7]:
ALLOWED_LABELS = set(annotations['allowed_labels'])

def annotation_pool(query_id):
    rows = [row for row in annotations['labels'] if row['query_id'] == query_id]
    return pd.DataFrame(rows)

def set_label(query_id, source_document_id, label, reviewer, note=None):
    if label not in ALLOWED_LABELS:
        raise ValueError(f'label must be one of {sorted(ALLOWED_LABELS)}')
    if not reviewer or not reviewer.strip():
        raise ValueError('reviewer is required for a human judgment')
    for row in annotations['labels']:
        if row['query_id'] == query_id and row['source_document_id'] == source_document_id:
            row.update(label=label, reviewer=reviewer.strip(), reviewed_at=datetime.now(timezone.utc).isoformat(), note=note)
            annotations_path.write_text(json.dumps(annotations, ensure_ascii=False, indent=2) + '\n')
            return row
    raise KeyError('Document is not in the pooled annotation set for this question')

display(annotation_pool(query_ids[0]))
# Example (uncomment after review):
# set_label(query_ids[0], 'pubmed:37822045', 'relevant', reviewer='Your name', note='Direct population/intervention/outcome match')

,query_id,source_document_id,pmid,label,reviewer,reviewed_at,note
0,GQ-SUP-061-01,pubmed:37822045,37822045,not_reviewed,None,None,None
1,GQ-SUP-061-01,pubmed:40911749,40911749,not_reviewed,None,None,None
2,GQ-SUP-061-01,pubmed:36546868,36546868,not_reviewed,None,None,None
3,GQ-SUP-061-01,pubmed:41879202,41879202,not_reviewed,None,None,None
4,GQ-SUP-061-01,pubmed:40425393,40425393,not_reviewed,None,None,None
5,GQ-SUP-061-01,pubmed:39914233,39914233,not_reviewed,None,None,None
6,GQ-SUP-061-01,pubmed:37551682,37551682,not_reviewed,None,None,None
7,GQ-SUP-061-01,pubmed:42478598,42478598,not_reviewed,None,None,None
8,GQ-SUP-061-01,pubmed:42879451,42879451,not_reviewed,None,None,None
9,GQ-SUP-061-01,pubmed:39444151,39444151,not_reviewed,None,None,None


## Recalculate metrics after annotation

After completing the pooled top-10 judgments for one or more questions, run the repository evaluation command documented in `docs/retrieval_experiment_report.md`, then rerun the loading cells. Until then, Recall@5, Recall@10, nDCG@10, and MRR remain unavailable by design.